Diagrama UML

In [3]:
![diagram_uml.png](attachment:diagram_uml.png)

'[diagram_uml.png]' is not recognized as an internal or external command,
operable program or batch file.


Web Scraping: the automated process of extracting data from websites

In [2]:
import requests                             # import requests para hacer peticiones HTTP
from bs4 import BeautifulSoup               # Beatiful para traer un bloque de código HTML y parsearlo
                                            #parsear: convertir un bloque de código HTML a un formato que podamos manipular con Python
from urllib.parse import urljoin            # permite extraer la ruta absoluta de un enlace relativo 
import random
import concurrent.futures                   # manejar hilos para ejecutar tareas en paralelo
import re                                   # para limpiar el precio de caracteres no numéricos        

url = "https://books.toscrape.com/"         

# Función para obtener el HTML parseado
def obtener_soup(url):
    respuesta = requests.get(url)
    
    #verficamos que la respuesta sea exitosa
    if respuesta.status_code != 200:
        print(f"Error al acceder a {url}")
        return None
    return BeautifulSoup(respuesta.text, "html.parser")

# Extraer todas las categorías
def extraer_categorias():
    soup = obtener_soup(url)                                    # obtenemos el soup de la página principal
    categorias = {}                                             # diccionario de las categorias
    
    contenedor = soup.find("div", class_="side_categories")     # buscamos la barra lateral donde se encuentran las categorias
    enlaces = contenedor.find_all("a")                          # obtenemos todos los enlaces dentro de esa barra lateral 
    
    for enlace in enlaces[1:]:                                  # Omitimos "All books"
        nombre = enlace.text.strip()                            # obtenemos el nombre de la categoría y eliminamos espacios
        href = enlace["href"]                                   # obtenemos el enlace de la categoria
        url_completa = urljoin(url, href)                       # unimos ese enlace para obtener la url completa
        
        categorias[nombre] = url_completa                       #guardamos el nombre de la categoria como clave y la url como valor                  
        print(f"Categoría encontrada: {nombre} -> {url_completa}")
    
    return categorias

# Extraer libros de una categoría
def extraer_libros_categoria(url_categoria, nombre_categoria):
    libros = []                                                 # lista para guardar los libros de una categoria
    url_actual = url_categoria                                  
    
    while True:
        soup = obtener_soup(url_actual)                         # obtenemos el soup de la página actual de la categoría
        
        if soup is None:
            break
        
        productos = soup.select(".product_pod")                 # buscamos los libros de la cateogoria
        
        # obtenemos el titulo, el precio y el rating de cada libro
        for producto in productos:
            titulo = producto.select_one("h3 a")["title"]       
            precio = producto.select_one(".price_color").get_text()
            rating = producto.select_one(".star-rating")["class"][1]
            
        # lo añadimos a la lista de libros con su respectiva categoría
            libros.append({
                "titulo": titulo,
                "precio": precio,
                "rating": rating,
                "categoria": nombre_categoria
            })
        
        # Buscar botón next
        siguiente = soup.select_one("li.next a")
        
        # Si existe el botón next, actualizamos la url para la siguiente página, sino rompemos el ciclo
        if siguiente:
            href_siguiente = siguiente["href"]
            url_actual = urljoin(url_actual, href_siguiente)
        else:
            break
    
    return libros

# Funcion para asignar autores aleatorios y limpiar el precio de los libros
def asignar_autores(libros):
    
    # lista de autores ficticios
    autores_fake = [
        "Adrián Valcárcel", "Lucía Montelara", "Tomás G. Ferreyra",
        "Elena Kovač", "Mateo Villalobos", "Camila Rousseau",
        "Santiago A. Duarte", "Irene Whitmore", "Hugo Delacroix",
        "Marina Cortázar", "Daniel K. Navarro", "Valentina Sforza",
        "Bruno Álvarez", "Clara M. Ionescu", "Sebastián Morante",
        "Amelia Hastings", "Julián Orlov", "Isabella Romano",
        "Thiago Mendes", "Sofía Arancibia", "Gabriel Lefèvre",
        "Olivia Blackwood", "Rodrigo Santillán", "Beatriz Roldán",
        "Tomás A. Serrano", "Isabel Thornton", "Joaquín Molina",
        "Héctor Salinas"
    ]
    
    # mapeamos las estrellas a numeros
    rating_map = {
        "One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5
    }
    
    # iteramos sobre los libros para asignar un id, un autor aleatorio, limpiar el precio y convertir el rating a número
    for i, libro in enumerate(libros, start=1):
        libro["id"] = i
        libro["autor"] = random.choice(autores_fake)
        
        # Limpiamosel precio eliminado cualquier caracter extraño 
        precio_sucio = libro["precio"]
        precio_solo_numeros = "".join(re.findall(r"[\d.]+", precio_sucio))        
        libro["precio"] = float(precio_solo_numeros)
        
        # pasamos el numero del rating como valor numerico en lugar de texto
        libro["rating"] = rating_map[libro["rating"]]
    
    return libros

#Funcion principal que hace el scrapeo completo del sitio
def scrapeo_completo():
    categorias = extraer_categorias()    # obtenemos las categorias del sitio
    todos_los_libros = []                # lista para guardar todos los libros de todas las categorias

# Usamos ThreadPoolExrcutor para ejecutas proceso paralelos en hilos diferentes
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        futuros = [executor.submit(extraer_libros_categoria, url_cat, nombre) 
                   for nombre, url_cat in categorias.items()] 
        print("Scrapeando categorías en paralelo...")
        
        # a medida que cada hilo termine, vamos obteniendo los libros de cada categoría y los vamos añadiendo a la lista total
        for futuro in concurrent.futures.as_completed(futuros):
            libros_categoria = futuro.result() 
            todos_los_libros.extend(libros_categoria)
            print(f"Categoría scrapeada, libros encontrados: {len(libros_categoria)}")
            
    # una vez que tenemos todos los libros, asignamos autores aleatorios y limpiamos el precio
    libros_finales = asignar_autores(todos_los_libros)
        
    return libros_finales, categorias

libros, categorias = scrapeo_completo()

print(f"\nTotal libros extraídos: {len(libros)}")
print(f"Total categorías scrapeadas: {len(categorias)}")

Categoría encontrada: Travel -> https://books.toscrape.com/catalogue/category/books/travel_2/index.html
Categoría encontrada: Mystery -> https://books.toscrape.com/catalogue/category/books/mystery_3/index.html
Categoría encontrada: Historical Fiction -> https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html
Categoría encontrada: Sequential Art -> https://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html
Categoría encontrada: Classics -> https://books.toscrape.com/catalogue/category/books/classics_6/index.html
Categoría encontrada: Philosophy -> https://books.toscrape.com/catalogue/category/books/philosophy_7/index.html
Categoría encontrada: Romance -> https://books.toscrape.com/catalogue/category/books/romance_8/index.html
Categoría encontrada: Womens Fiction -> https://books.toscrape.com/catalogue/category/books/womens-fiction_9/index.html
Categoría encontrada: Fiction -> https://books.toscrape.com/catalogue/category/books/fiction_10/

DDL (CREATE TABLE) y DML (INSERT INTO)

In [ ]:
import sqlite3                                                  # importamos sqlite3 para manejar la base de datos sqlite

conn = sqlite3.connect("libros.db")                             # creamos la conexion a la bd 
cursor = conn.cursor()                                          # creamos un objeto llamado cursor para ejecutar comandos

cursor.execute("PRAGMA foreign_keys = ON")                      # habilitamos las claves foraneas para que funcionen las relaciones entre tablas

# borramos las tablas si ya existen para evitar errores al crear las tablas de nuevo
cursor.execute("DROP TABLE IF EXISTS libros_autores")
cursor.execute("DROP TABLE IF EXISTS libros")
cursor.execute("DROP TABLE IF EXISTS autores")
cursor.execute("DROP TABLE IF EXISTS categorias")

# Creamos la tabla categorias
cursor.execute("""
CREATE TABLE categorias (
    id INTEGER PRIMARY KEY,
    nombre_categoria TEXT UNIQUE,
    url TEXT
)
""")

# Creamoa la tabla autores
cursor.execute("""
CREATE TABLE autores (
    id INTEGER PRIMARY KEY,
    nombre TEXT UNIQUE
)
""")

#Creamos la tabla libros con una clave foranea a categorias
cursor.execute("""
CREATE TABLE libros (
    id INTEGER PRIMARY KEY,
    titulo TEXT,
    precio REAL,
    rating INTEGER,
    url TEXT,  
    categoria_id INTEGER,
    FOREIGN KEY (categoria_id) REFERENCES categorias(id) ON DELETE CASCADE
)
""")


# Creamos la tabla intermedia con llaves foraneas a libros y autores para manejar la relacion muchos a muchos
cursor.execute("""
CREATE TABLE libros_autores (
    libro_id INTEGER,
    autor_id INTEGER,
    PRIMARY KEY (libro_id, autor_id),
    FOREIGN KEY (libro_id) REFERENCES libros(id) ON DELETE CASCADE,
    FOREIGN KEY (autor_id) REFERENCES autores(id) ON DELETE CASCADE
)
""")

conn.commit()  # guardamos los cambios en la base de datos
print("Base de datos creada correctamente y limpia.")

In [ ]:
# Bloque para insertar los datos en la base de datos

# diccionario para mapear la categoria e ingresar los datos en la tabla categorias
categorias_diccionario = {}

#  iteramos sobre las categorias para asignar un id a cada una y guardarlas en el diccionario
for i, nombre in enumerate(categorias.keys(), start= 1):
    categorias_diccionario[nombre] = i

# preparamos los datos de las categorias para insertarlos en la tabla categorias
datos_categorias = [(id_categoria, nombre) for nombre, id_categoria in categorias_diccionario.items()]

# insertamos las categorias en la tabla categorias
cursor.executemany(
    "INSERT INTO categorias (id, nombre_categoria) VALUES (?, ?);",
    datos_categorias)

# listamos los autores unicos para crear la tabla autores y la tabla intermedia libros_autores
autores_unicos = list(set([libro["autor"] for libro in libros]))
autores_diccionario = {}

#  iteramos sobre los autores para asignar un id a cada uno y guardarlos en el diccionario
for i, nombre in enumerate(autores_unicos, start= 1):
    autores_diccionario[nombre] = i

# preparamos los datos de los autores para insertarlos en la tabla autores
datos_autores = [(id_autor, nombre) for nombre, id_autor in autores_diccionario.items()]

# insertamos los autores en la tabla autores
cursor.executemany(
    "INSERT INTO autores (id, nombre) VALUES (?, ?);",
    datos_autores)

# lista para guardar lo datos de los libros
datos_libro = []

# iteramos sobre los libros para preparar los datos y luego insertarlos en la tabla libros
for libro in libros:
    datos_libro.append((libro["id"], 
                        libro["titulo"],
                        libro["precio"],
                        libro["rating"],
                        libro.get("url", None),
                        categorias_diccionario[libro["categoria"]]
                        ))    

# insertamos los libros en la tabla libros
cursor.executemany(
    "INSERT INTO libros (id, titulo, precio, rating, url, categoria_id) VALUES (?, ?, ?, ?, ?, ?);",
    datos_libro)

#  lista para guardar los datos de la tabla intermedia libros_autores
datos_libro_autores = []

# iteramos sobre los libros para preparar los datos de la tabla intermedia libros_autores y luego insertarlos
for libro in libros:
    datos_libro_autores.append((libro["id"], # libro_id
                                autores_diccionario[libro["autor"]] # autor_id
                                ))

# insertamos las llaves foraneas en la tabla libros_autores
cursor.executemany(
    "INSERT INTO libros_autores (libro_id, autor_id) VALUES (?, ?);",
    datos_libro_autores)

# guardamos los cambios en la base de datos
conn.commit()
print("datos insertados sin errores")
conn.close()